# Sync Full Load Bronze Tables & Landing Volumes in Unity Catalog
**Framework:** Metadata-Driven Pipeline Framework  
**Purpose:** Runs after ADF Full Load Copy activity to ensure Unity Catalog Landing Volumes exist and drops & recreates external tables on the Bronze layer pointing to the newly written Parquet files.

In [ ]:
# 1. Define Input Widgets
import json

dbutils.widgets.text("source_table_name", "ALL", "Source Table Name (CoverageTypes, Agents, or ALL)")
dbutils.widgets.text("target_catalog", "bronze_catalog", "Target Catalog")
dbutils.widgets.text("target_schema", "raw_data", "Target Schema")
dbutils.widgets.text("volume_catalog", "landing_catalog", "Volume Catalog")
dbutils.widgets.text("volume_schema", "source_sql", "Volume Schema")
dbutils.widgets.text("landing_container", "landing", "Landing Container")
dbutils.widgets.text("bronze_container", "bronze", "Bronze Container")
dbutils.widgets.text("storage_account_name", "azureprojectdevadls01", "Storage Account Name")
dbutils.widgets.text("folder_prefix", "home/datalake/claims", "Folder Prefix")

source_table_name    = dbutils.widgets.get("source_table_name").strip()
target_catalog       = dbutils.widgets.get("target_catalog").strip()
target_schema        = dbutils.widgets.get("target_schema").strip()
volume_catalog       = dbutils.widgets.get("volume_catalog").strip()
volume_schema        = dbutils.widgets.get("volume_schema").strip()
landing_container    = dbutils.widgets.get("landing_container").strip()
bronze_container     = dbutils.widgets.get("bronze_container").strip()
storage_account_name = dbutils.widgets.get("storage_account_name").strip()
folder_prefix        = dbutils.widgets.get("folder_prefix").strip().strip("/")

print(f"Sync Target Table: {source_table_name}")
print(f"Target Catalog.Schema: `{target_catalog}`.`{target_schema}`")
print(f"Volume Catalog.Schema: `{volume_catalog}`.`{volume_schema}`")
print(f"Landing Container: {landing_container}")
print(f"Bronze Container: {bronze_container}")
print(f"Storage Account: {storage_account_name}")
print(f"Folder Prefix: {folder_prefix}")

In [ ]:
# 2. Determine Tables to Process
all_full_load_tables = ["CoverageTypes", "Agents"]

if not source_table_name or source_table_name.upper() == "ALL":
    tables_to_sync = all_full_load_tables
else:
    tables_to_sync = [t.strip() for t in source_table_name.split(",") if t.strip()]

print(f"Tables scheduled for Bronze sync: {tables_to_sync}")

In [ ]:
# 3. Ensure Catalogs & Schemas Exist
print(f"Ensuring Volume Schema `{volume_catalog}`.`{volume_schema}` exists...")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{volume_catalog}`.`{volume_schema}`")

print(f"Ensuring Target Schema `{target_catalog}`.`{target_schema}` exists...")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{target_catalog}`.`{target_schema}`")

In [ ]:
# 4. Process Each Full-Load Table: Create Volume + Drop & Recreate Bronze External Table
sync_results = {}

for tbl in tables_to_sync:
    print("=" * 70)
    print(f"Processing Full-Load Table: {tbl}")
    print("=" * 70)
    
    # 4.1 Create External Volume on Landing Container if not exists
    vol_name = f"vol_{tbl.lower()}"
    vol_location = f"abfss://{landing_container}@{storage_account_name}.dfs.core.windows.net/{folder_prefix}/{tbl}"
    create_vol_sql = f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS `{volume_catalog}`.`{volume_schema}`.`{vol_name}`
    LOCATION '{vol_location}'
    COMMENT 'External landing volume for {tbl} table'
    """
    try:
        spark.sql(create_vol_sql)
        print(f"[VOLUME SUCCESS] `{volume_catalog}`.`{volume_schema}`.`{vol_name}` registered at {vol_location}")
    except Exception as e_vol:
        print(f"[VOLUME WARNING] Volume creation note for `{vol_name}`: {str(e_vol)}")
    
    # 4.2 Drop and Recreate Bronze External Parquet Table
    bronze_location = f"abfss://{bronze_container}@{storage_account_name}.dfs.core.windows.net/{folder_prefix}/{tbl}"
    full_table_name = f"`{target_catalog}`.`{target_schema}`.`{tbl}`"
    
    drop_table_sql = f"DROP TABLE IF EXISTS {full_table_name}"
    spark.sql(drop_table_sql)
    print(f"[TABLE DROP] Dropped existing table {full_table_name} if present.")
    
    create_table_sql = f"""
    CREATE TABLE {full_table_name}
    USING PARQUET
    LOCATION '{bronze_location}'
    COMMENT 'Bronze external Parquet table for full-load {tbl}'
    """
    spark.sql(create_table_sql)
    print(f"[TABLE CREATE] Recreated {full_table_name} pointing to {bronze_location}")
    
    # 4.3 Verify Record Count
    try:
        row_count = spark.table(full_table_name).count()
        print(f"[TABLE VERIFY] Table {full_table_name} contains {row_count} records.")
        sync_results[tbl] = {"status": "Success", "row_count": row_count, "location": bronze_location}
    except Exception as e_verify:
        print(f"[TABLE WARNING] Could not read record count: {str(e_verify)}")
        sync_results[tbl] = {"status": "Created_Unverified", "error": str(e_verify), "location": bronze_location}

print("=" * 70)
print("Full Load Bronze Sync Summary:")
for tbl, res in sync_results.items():
    print(f" - {tbl}: {res}")
print("=" * 70)

In [ ]:
# 5. Exit Notebook with Output
total_rows = sum([res.get("row_count", 0) for res in sync_results.values() if isinstance(res.get("row_count"), int)])
exit_payload = {
    "status": "Success",
    "tables_synced": tables_to_sync,
    "total_tables": len(tables_to_sync),
    "total_rows_verified": total_rows,
    "results": sync_results
}

dbutils.notebook.exit(json.dumps(exit_payload))